# Baseline Experiments

Compare text dataset distillation methods on classification.

For each method the notebook does the same five steps — visible in the helper below:

1. **Data** — `load_baseline_data(dataset_name, seed=SEED)`
2. **Selection** — `select_*(...)` (the method that makes this baseline different)
3. **Model** — `load_tokenizer(...)` + `load_sequence_classifier(...)`
4. **Training** — `train_text_classifier(model=..., tokenizer=..., ...)`
5. **Save** — `save_baseline_run(...)`

Selection runs once per dataset; training runs once per (dataset, model).

In [ ]:
from pathlib import Path

from text_distillation import (
    TimingTracker,
    load_baseline_data,
    save_baseline_run,
)
from text_distillation.analysis import collect_runs
from text_distillation.distillation import (
    select_herding,
    select_kcenter_embeddings,
    select_kcenter_tfidf,
    select_random,
    select_stratified_random,
)
from text_distillation.model import (
    load_sequence_classifier,
    load_tokenizer,
    train_text_classifier,
)
from text_distillation.saving import create_run_dir

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS_DIR = PROJECT_ROOT / "artifacts" / "runs"

In [ ]:
# Shared across all baselines.
DATASET_NAMES = ["ag_news", "sst2", "qqp", "mnli-m"]
MODEL_NAMES = [
    "bert-base-uncased",
    "roberta-base",
    "albert-base-v2",
    "microsoft/deberta-v3-base",
    "bert-large-uncased",
    "xlnet-base-cased",
]
SEED = 42

# Used by selection methods (all except full_data).
K_PER_CLASS = 100

# Encoder used by kcenter_cls / herding to embed the train pool.
EMBEDDING_MODEL_NAME = "bert-base-uncased"

In [ ]:
def run_all_models(data, train_dataset, method_name, k_per_class, run_suffix_str, selection_tracker):
    """For one (dataset, method, distilled subset): train each model, save each run.

    The model-loading step is inside the loop so each model gets a fresh classifier
    head sized for `data.num_labels`. Timings from `selection_tracker` (the same
    selection was used for every model on this dataset) are merged into each
    per-model tracker.
    """
    rows = []
    safe_dataset = data.dataset_info.name.replace("-", "_")
    for model_name in MODEL_NAMES:
        safe_model = model_name.replace("/", "_").replace("-", "_")
        run_dir = create_run_dir(RUNS_DIR, f"{method_name}_{safe_dataset}_{safe_model}_{run_suffix_str}")

        tracker = TimingTracker()
        tracker.merge(selection_tracker)

        # 3. Model — load tokenizer + classifier head sized for this dataset.
        tokenizer = load_tokenizer(model_name)
        model = load_sequence_classifier(
            model_name,
            num_labels=data.num_labels,
            label_names=data.label_names,
        )

        # 4. Training.
        _, metrics = train_text_classifier(
            model=model,
            tokenizer=tokenizer,
            train_dataset=train_dataset,
            eval_dataset=data.eval_dataset,
            output_dir=run_dir,
            text_columns=data.dataset_info.text_columns,
            metric_name=data.dataset_info.metric_name,
            seed=SEED,
            tracker=tracker,
        )

        # 5. Save config + metrics + distilled dataset.
        row = save_baseline_run(
            run_dir=run_dir,
            data=data,
            method_name=method_name,
            model_name=model_name,
            k_per_class=k_per_class,
            seed=SEED,
            train_dataset=train_dataset,
            metrics=metrics,
            tracker=tracker,
            project_root=PROJECT_ROOT,
        )
        rows.append(row)
        print(f"    [{model_name}] accuracy={metrics['accuracy']:.3f} f1_macro={metrics['f1_macro']:.3f}")
    return rows


all_results = {}

## 01 Full-Data Baseline

Train each model on the full train split per dataset — establishes the upper bound.

In [ ]:
method_name = "full_data"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} (no selection)")

    # 2. Selection — no-op: train on the full pool.
    train_dataset = data.train_pool
    selection_tracker = TimingTracker()

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, None, "full", selection_tracker)
    )

## 02 Random Coreset Baseline

Sample `K_PER_CLASS * num_labels` random examples (no class balancing).

In [ ]:
method_name = "random"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    k_total = K_PER_CLASS * data.num_labels
    print(f"[{dataset_name}] {method_name} k={k_total}")

    # 2. Selection — one random subset, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_random(data.train_pool, k=k_total, seed=SEED)

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker)
    )

## 03 Stratified Random Coreset Baseline

Sample `K_PER_CLASS` random examples per class — class-balanced version of 02.

In [ ]:
method_name = "stratified_random"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class")

    # 2. Selection — K_PER_CLASS per class, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_stratified_random(
            data.train_pool,
            k_per_class=K_PER_CLASS,
            label_column=data.dataset_info.label_column,
            seed=SEED,
        )

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker)
    )

## 04 K-Center over TF-IDF Baseline

Greedy k-center per class on TF-IDF features — picks examples that cover the lexical space.

In [ ]:
method_name = "kcenter_tfidf"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class")

    # 2. Selection — k-center over TF-IDF, reused across all models.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_kcenter_tfidf(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            seed=SEED,
        )

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker)
    )

## 05 K-Center over Encoder Embeddings Baseline

Greedy k-center per class on pooled encoder embeddings — picks examples that cover the semantic space. Pooling is auto-selected per model (XLNet → last token).

In [ ]:
method_name = "kcenter_cls"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class (embedder={EMBEDDING_MODEL_NAME})")

    # 2. Selection — embed pool once with EMBEDDING_MODEL_NAME, then k-center per class.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_kcenter_embeddings(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            model_name=EMBEDDING_MODEL_NAME,
            seed=SEED,
            tracker=selection_tracker,
        )

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker)
    )

## 06 Herding Baseline

Greedy herding (Welling 2009) over pooled encoder embeddings — picks examples whose running mean best approximates the class mean.

In [ ]:
method_name = "herding"
all_results[method_name] = []
for dataset_name in DATASET_NAMES:
    # 1. Data.
    data = load_baseline_data(dataset_name, seed=SEED)
    print(f"[{dataset_name}] {method_name} {K_PER_CLASS}/class (embedder={EMBEDDING_MODEL_NAME})")

    # 2. Selection — embed pool once with EMBEDDING_MODEL_NAME, then herding per class.
    selection_tracker = TimingTracker()
    with selection_tracker.measure("selection_sec"):
        train_dataset = select_herding(
            data.train_pool,
            text_columns=data.dataset_info.text_columns,
            label_column=data.dataset_info.label_column,
            k_per_class=K_PER_CLASS,
            model_name=EMBEDDING_MODEL_NAME,
            seed=SEED,
            tracker=selection_tracker,
        )

    all_results[method_name].extend(
        run_all_models(data, train_dataset, method_name, K_PER_CLASS, f"k{K_PER_CLASS}", selection_tracker)
    )

## Aggregated results

Load every run from `artifacts/runs/` into a single DataFrame.

In [ ]:
df = collect_runs(RUNS_DIR)
df[["experiment_name", "method_name", "model_name", "dataset_name", "k_total",
    "accuracy", "f1_macro", "timings_selection_sec", "timings_training_sec"]].sort_values(
    ["dataset_name", "method_name", "model_name"]
)